# CarePath DARAG GEC Colab

This notebook follows the DARAG path: real Gipformer pairs first, open-weight synthetic clean transcripts, TTS audio, Gipformer synthetic ASR pairs, then QLoRA GEC. CKey is not used for synthetic transcript generation.

## VSCode Colab Extension — quick setup

The file-upload widget (`files.upload()`) does **not** render in VSCode. Before running **Cell 2**, uncomment **one** line in the config cell below it and set the correct path:

| Env var | When to use |
|---|---|
| `CAREPATH_REPO_DIR` | Repo already on disk (most common — clone to `/content/carepath` first) |
| `CAREPATH_REPO_ZIP` | You have a `.zip` of the repo |
| `CAREPATH_REPO_URL` | Clone from a git URL on the fly |

In [ ]:
!nvidia-smi
!pip install -q "datasets[audio]" soundfile huggingface_hub sherpa-onnx numpy jiwer evaluate
!pip install -q transformers accelerate peft trl bitsandbytes sentencepiece

In [ ]:
import os

# ── VSCode / Colab-Extension: set the repo source ─────────────────────────────
# The repo is private. We read GITHUB_TOKEN from Colab Secrets (the 🔑 icon in
# the left sidebar). Add a secret named GITHUB_TOKEN with a classic PAT that
# has the "repo" scope, then enable "Notebook access" for it.
try:
    from google.colab import userdata
    _token = userdata.get("GITHUB_TOKEN")
    os.environ["CAREPATH_REPO_URL"] = f"https://{_token}@github.com/truong-tt/carepath"
    print("GITHUB_TOKEN loaded from Colab Secrets.")
except Exception:
    # Fallback: paste your token directly (do NOT commit this value to git)
    # os.environ["CAREPATH_REPO_URL"] = "https://YOUR_TOKEN_HERE@github.com/truong-tt/carepath"
    raise RuntimeError(
        "Could not load GITHUB_TOKEN from Colab Secrets.\n"
        "Add it via the 🔑 Secrets panel (left sidebar) and enable Notebook access,\n"
        "or uncomment the fallback line above and paste your token."
    )
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

def is_carepath_repo(path: Path) -> bool:
    return (path / 'scripts' / 'create_gec_pairs.py').exists() and (path / 'apps' / 'api' / 'hopegait').exists()

def find_carepath_repo(base: Path) -> Path | None:
    if is_carepath_repo(base):
        return base
    if not base.exists():
        return None
    for marker in base.rglob('scripts/create_gec_pairs.py'):
        candidate = marker.parents[1]
        if is_carepath_repo(candidate):
            return candidate
    return None

def extract_repo_zip(zip_path: Path, extract_dir: Path = Path('/content/carepath_upload')) -> Path | None:
    if not zip_path.exists():
        return None
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(extract_dir)
    return find_carepath_repo(extract_dir)

candidates = [
    Path(os.environ.get('CAREPATH_REPO_DIR', '/content/carepath')),
    Path('/content/carepath'),
    Path('/content/CarePath'),
    Path('/content/drive/MyDrive/carepath'),
    Path.cwd(),
]

zip_candidates = [
    Path(value)
    for value in [
        os.environ.get('CAREPATH_REPO_ZIP'),
        '/content/carepath.zip',
        '/content/CarePath.zip',
        '/content/drive/MyDrive/carepath.zip',
        '/content/drive/MyDrive/CarePath.zip',
    ]
    if value
]

REPO_DIR = next((repo for path in candidates if (repo := find_carepath_repo(path))), None)

if REPO_DIR is None:
    REPO_DIR = next((repo for path in zip_candidates if (repo := extract_repo_zip(path))), None)

if REPO_DIR is None:
    repo_url = os.environ.get('CAREPATH_REPO_URL')
    target_dir = Path(os.environ.get('CAREPATH_REPO_DIR', '/content/carepath'))
    if repo_url:
        subprocess.run(['git', 'clone', repo_url, str(target_dir)], check=True)
        REPO_DIR = find_carepath_repo(target_dir)
    else:
        try:
            from google.colab import files
        except ImportError as exc:
            raise RuntimeError(
                'CarePath repo not found. Set CAREPATH_REPO_DIR, CAREPATH_REPO_ZIP, or CAREPATH_REPO_URL.'
            ) from exc
        print('CarePath repo not found. Upload a .zip of the CarePath/carepath repository now.')
        print('If the upload widget is unavailable, upload carepath.zip to /content with the left Files panel, then rerun this cell.')
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
        if not zip_names:
            raise RuntimeError('No .zip file uploaded. Zip the repository folder and rerun this cell.')
        REPO_DIR = extract_repo_zip(Path(zip_names[0]))

if REPO_DIR is None or not is_carepath_repo(REPO_DIR):
    raise RuntimeError(f'Not a CarePath repo: {REPO_DIR}')

print(f'Using CarePath repo: {REPO_DIR}')
%cd {REPO_DIR}

## 1. Retrieval And Real Pair Smoke Run

In [ ]:
!PYTHONPATH=apps/api python scripts/build_term_datastore.py \
  --dataset tensorxt/ViMedCSS \
  --limit-per-split 20 \
  --output artifacts/retrieval/term_datastore_smoke.json

!PYTHONPATH=apps/api python scripts/create_gec_pairs.py \
  --output artifacts/gec_pairs/vimedcss_gipformer_pairs_smoke.jsonl \
  --limit-per-split 20 \
  --lexicon artifacts/retrieval/term_datastore_smoke.json \
  --resume

!PYTHONPATH=apps/api python scripts/evaluate_corrections.py \
  --input artifacts/gec_pairs/vimedcss_gipformer_pairs_smoke.jsonl \
  --prediction-column raw_asr

## 2. Open-Weight Synthetic Clean Transcripts

In [ ]:
!PYTHONPATH=apps/api python scripts/generate_synthetic_transcripts.py \
  --pairs artifacts/gec_pairs/vimedcss_gipformer_pairs_smoke.jsonl \
  --output artifacts/synthetic/synthetic_clean_smoke.jsonl \
  --model Qwen/Qwen3-4B-Instruct-2507 \
  --count 50 \
  --load-in-4bit

## 3. TTS Audio And Synthetic Gipformer Pairs

In [ ]:
!PYTHONPATH=apps/api python scripts/synthesize_speech.py \
  --input artifacts/synthetic/synthetic_clean_smoke.jsonl \
  --output artifacts/synthetic/synthetic_audio_manifest_smoke.jsonl \
  --limit 10 \
  --resume

!PYTHONPATH=apps/api python scripts/create_synthetic_gec_pairs.py \
  --input artifacts/synthetic/synthetic_audio_manifest_smoke.jsonl \
  --output artifacts/gec_pairs/darag_synthetic_pairs_smoke.jsonl \
  --limit 10 \
  --resume

## 4. Train Only After Baselines

Do not train until raw Gipformer and CKey/RAG baseline metrics exist. Do not ship trained GEC unless validation and hard split term F1 improve without degrading number/unit preservation.

In [ ]:
!PYTHONPATH=apps/api python scripts/train_gec_lora.py \
  --pairs artifacts/gec_pairs/vimedcss_gipformer_pairs_smoke.jsonl \
  --output-dir artifacts/gec_lora/qwen3_gec_smoke \
  --max-steps 20